# Этап 7: KAN vs 1D CNN (M0/M1/M2)

**Цель:** сравнить IrResnet4 (M0), IrKanHybrid (M1), IrKanNet (M2).

**Вход:** `dataset_v003` (+ опционально `external_sdbs/sdbs_eval.npz`).

**Colab full:** выполните A + A2 + C. **Local:** B + C (`paths.local.yaml`).

## Выбор среды (выполните ОДНУ ячейку)

| Среда | Запустить | Пропустить |
|-------|-----------|------------|
| **Google Colab** | **A. Colab** (+ при полном датасете **A2. Drive**) | **B. Local** |
| **Локальный Jupyter** | **B. Local** | **A** и **A2** (включая `drive.mount`) |

После A или B выполните **C. Пути и данные**.
Подробнее: [`docs/NOTEBOOKS.md`](../docs/NOTEBOOKS.md).


In [ ]:
# === A. Colab: окружение ===
# Локально эту ячейку НЕ запускайте (см. B. Local).
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/Lamblador/IR_expert_system_3.git'
REPO_BRANCH = 'colab-v1'
REPO_DIR = Path('/content/IR_expert_system_3')

def _run_git(cmd, cwd=None):
    print('git', ' '.join(cmd))
    subprocess.run(cmd, cwd=cwd, check=True)

if (REPO_DIR / '.git').is_dir():
    _run_git(['git', 'fetch', 'origin', REPO_BRANCH], cwd=REPO_DIR)
    _run_git(['git', 'checkout', REPO_BRANCH], cwd=REPO_DIR)
    _run_git(['git', 'pull', '--ff-only', 'origin', REPO_BRANCH], cwd=REPO_DIR)
else:
    if REPO_DIR.exists():
        raise RuntimeError(f'{REPO_DIR} существует, но это не git-репозиторий')
    _run_git([
        'git', 'clone', '-b', REPO_BRANCH, '--single-branch',
        REPO_URL, str(REPO_DIR),
    ])

ROOT = REPO_DIR.resolve()
os.chdir(ROOT)
try:
    from IPython import get_ipython
    get_ipython().run_line_magic('cd', str(ROOT))
except Exception:
    pass
rev = subprocess.check_output(
    ['git', 'rev-parse', '--short', 'HEAD'], cwd=ROOT, text=True
).strip()
print(f'ROOT: {ROOT} @ {REPO_BRANCH} ({rev})')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[torch]'], check=True)
print('pip install OK')
IR_ENV = 'colab'


### A2. Google Drive (только Colab full)

Пропустите для HF smoke и локально.

In [ ]:
# === A2. Colab Drive (full dataset) ===
# Нужен только для полного датасета на Google Drive.
# Для HF smoke (dataset_mini) эту ячейку ПРОПУСТИТЕ.
# Локально НЕ запускайте.
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
IR_DATA = Path('/content/drive/MyDrive/ir_data')
RUNS_DRIVE = Path('/content/drive/MyDrive/ir_expert_system_3/runs')
RUNS_DRIVE.mkdir(parents=True, exist_ok=True)
print('IR_DATA exists:', IR_DATA.exists(), IR_DATA)
print('RUNS_DRIVE:', RUNS_DRIVE)


### B. Локальный Jupyter

Пропустите в Colab.

In [ ]:
# === B. Local: окружение ===
# В Google Colab эту ячейку НЕ запускайте (см. A. Colab).
import os
import subprocess
import sys
from pathlib import Path

def _find_repo_root(start: Path) -> Path:
    """Walk-up до каталога с pyproject.toml (фикс nested-clone из notebooks/)."""
    cur = start.resolve()
    for p in [cur, *cur.parents]:
        if (p / 'pyproject.toml').is_file():
            return p
    raise FileNotFoundError(
        'Не найден pyproject.toml выше cwd. '
        'Откройте ноутбук из клона репозитория или cd в корень IR_expert_system_3.'
    )

ROOT = _find_repo_root(Path.cwd())
os.chdir(ROOT)
try:
    from IPython import get_ipython
    get_ipython().run_line_magic('cd', str(ROOT))
except Exception:
    pass
print('ROOT (local, ветку не переключаем):', ROOT)

FORCE_REINSTALL = False  # True — принудительно pip install -e .[torch]
need_install = FORCE_REINSTALL
if not need_install:
    try:
        import ir_pipeline  # noqa: F401
    except ImportError:
        need_install = True
if need_install:
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[torch]'],
        check=True,
    )
    print('pip install OK')
else:
    print('ir_pipeline уже установлен — pip пропущен (FORCE_REINSTALL=True для переустановки)')
IR_ENV = 'local'


### C. Пути и данные

После A или B.

In [ ]:
# === C. Пути и данные ===
# Выполните после A или B. Контракт: ROOT, PATHS_YAML, paths, DATASET_DIR, BANDS_YAML, RUNS_DIR
import os
from pathlib import Path
from ir_pipeline.config_loader import load_yaml, resolve_paths

SMOKE_VERSION = 'dataset_mini'
FULL_VERSION = 'dataset_v003'
# Режим данных (если IR_ENV не задан — auto):
#   'local'       — configs/paths.local.yaml
#   'colab_smoke' — HF mini, paths.huggingface.yaml
#   'colab_full'  — Drive, paths.colab.yaml
DATA_MODE = 'auto'  # или 'local' | 'colab_smoke' | 'colab_full'

if 'IR_ENV' not in globals():
    IR_ENV = 'colab' if Path('/content').exists() else 'local'

if DATA_MODE == 'auto':
    if IR_ENV == 'local':
        DATA_MODE = 'local'
    elif 'IR_DATA' in globals() and Path(IR_DATA).exists():
        DATA_MODE = 'colab_full'
    else:
        DATA_MODE = 'colab_smoke'

if DATA_MODE == 'local':
    PATHS_YAML = Path('configs/paths.local.yaml')
    if not PATHS_YAML.is_file():
        raise FileNotFoundError(
            'Нет configs/paths.local.yaml — скопируйте configs/paths.local.example.yaml '
            'и пропишите raw_jcamp_dir / processed_root.'
        )
elif DATA_MODE == 'colab_full':
    if 'IR_DATA' not in globals():
        raise RuntimeError('Colab full: сначала выполните A2 (Drive mount) → IR_DATA')
    os.environ['IR_PROCESSED_ROOT'] = str(Path(IR_DATA) / 'processed')
    PATHS_YAML = Path('configs/paths.colab.yaml')
elif DATA_MODE == 'colab_smoke':
    PATHS_YAML = Path('configs/paths.huggingface.yaml')
else:
    raise ValueError(f'Неизвестный DATA_MODE={DATA_MODE!r}')

paths_cfg = load_yaml(PATHS_YAML)
if DATA_MODE == 'colab_smoke':
    paths_cfg['dataset_version'] = SMOKE_VERSION
    paths_cfg.pop('dataset_profile', None)
elif DATA_MODE == 'colab_full':
    paths_cfg['dataset_version'] = paths_cfg.get('dataset_version') or FULL_VERSION
# local: dataset_version / profile из yaml

paths = resolve_paths(paths_cfg)
DATASET_DIR = paths['processed_root'] / str(paths['dataset_version'])
BANDS_YAML = paths['bands_config']
RUNS_DIR = Path('runs')
RUNS_DIR.mkdir(parents=True, exist_ok=True)

spectra = DATASET_DIR / 'spectra.npz'
if not spectra.is_file():
    raise FileNotFoundError(
        f'Нет {spectra}. DATA_MODE={DATA_MODE}, PATHS_YAML={PATHS_YAML}'
    )
print('DATA_MODE:', DATA_MODE)
print('PATHS_YAML:', PATHS_YAML)
print('DATASET_DIR:', DATASET_DIR)
print('dataset_version:', paths['dataset_version'])
print('raw_jcamp_dir:', paths['raw_jcamp_dir'])


In [ ]:
# Пути SDBS holdout и runs на Drive (если A2 выполнен)
from pathlib import Path

if 'IR_DATA' in globals():
    SDSBS_EVAL = Path(IR_DATA) / 'external_sdbs' / 'sdbs_eval.npz'
    if 'RUNS_DRIVE' not in globals():
        RUNS_DRIVE = Path('/content/drive/MyDrive/ir_expert_system_3/runs')
else:
    SDSBS_EVAL = Path('data/external_sdbs/sdbs_eval.npz')
    RUNS_DRIVE = None
print('SDSBS_EVAL', SDSBS_EVAL, SDSBS_EVAL.is_file())
assert (DATASET_DIR / 'model_inputs.npz').is_file(), DATASET_DIR


## Данные и DataLoader (smoke test)

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch.utils.data import DataLoader

from ir_pipeline.dataset_preview import build_multilabel_matrix
from ir_pipeline.dataset_split import load_split_ids
from ir_pipeline.irresnet_train import IrDataset
from ir_pipeline.resnet_input import load_model_inputs

X_in, wn, spec_ids, X_ctx, ctx_cols = load_model_inputs(DATASET_DIR)
Y, class_names = build_multilabel_matrix(
    DATASET_DIR, spec_ids, BANDS_YAML, label_schema='structure_smarts'
)
splits = load_split_ids(DATASET_DIR)
tr_idx = [i for i, s in enumerate(spec_ids) if s in splits['train']]
ds = IrDataset(X_in[tr_idx[:64]], Y[tr_idx[:64]], X_ctx[tr_idx[:64]] if X_ctx is not None else None)
dl = DataLoader(ds, batch_size=8, shuffle=True)
xb, cb, yb = next(iter(dl))
print('batch X', xb.shape, 'context', cb.shape, 'Y', yb.shape, 'positives', yb.sum().item())

i0 = tr_idx[0]
fig, ax = plt.subplots(1, 3, figsize=(12, 3))
for ch, title in enumerate(['wavenumbers', 'absorption', 'peaks']):
    ax[ch].plot(wn, X_in[i0, ch])
    ax[ch].set_title(title)
    ax[ch].invert_xaxis()
fig.tight_layout()
plt.show()


## Модели M0/M1/M2

Код из пакета (`ir_pipeline.models`). Правки: `src/ir_pipeline/models/` + `pip install -e .[torch]`.

In [ ]:
from ir_pipeline.models.model_factory import build_spectrum_model, MODEL_FAMILIES

print('families:', MODEL_FAMILIES)


## Обучение одной модели и сравнение всех трёх

In [ ]:
import json
import shutil
from copy import deepcopy
from pathlib import Path
import pandas as pd
from ir_pipeline.config_loader import load_yaml, merge_train_defaults
from ir_pipeline.irresnet_train import train_irresnet_run

TRAIN_CFG = merge_train_defaults(load_yaml(Path('configs/train_kan_colab.yaml')))
HIDDEN_SIZE = int(TRAIN_CFG.get('compare_hidden_size', TRAIN_CFG.get('ir_hidden_size', 34)))
RUN_TAG = f'kan_cmp_h{HIDDEN_SIZE}'

FAMILY_LABELS = {
    'irresnet4': 'M0',
    'kan_hybrid': 'M1',
    'kan_full': 'M2',
}

def train_all_models(
    hidden_size: int,
    *,
    dataset_dir: Path,
    run_root: Path,
    train_cfg: dict,
    model_families: tuple[str, ...] = ('irresnet4', 'kan_hybrid', 'kan_full'),
    save_to_drive: Path | None = None,
    download_zip: bool = False,
) -> pd.DataFrame:
    run_root.mkdir(parents=True, exist_ok=True)
    summaries, histories, run_dirs = [], {}, {}
    for fam in model_families:
        label = FAMILY_LABELS.get(fam, fam)
        sub = run_root / f'{label.lower()}_{fam}'
        cfg = deepcopy(train_cfg)
        cfg['model_family'] = fam
        cfg['ir_hidden_size'] = hidden_size
        cfg['kan_full_hidden_size'] = hidden_size
        print('===', label, fam, 'hidden=', hidden_size, '=>', sub)
        summary = train_irresnet_run(
            dataset_dir=dataset_dir,
            run_dir=sub,
            bands_yaml=BANDS_YAML,
            train_cfg=cfg,
        )
        summary['model_label'] = label
        summaries.append(summary)
        run_dirs[label] = sub
        hist_path = sub / 'irresnet_history.json'
        if hist_path.is_file():
            histories[label] = json.loads(hist_path.read_text(encoding='utf-8'))
    df = pd.DataFrame(summaries)
    (run_root / 'compare_summary.json').write_text(
        df.to_json(orient='records', indent=2), encoding='utf-8'
    )
    if save_to_drive:
        dest = save_to_drive / run_root.name
        if dest.exists():
            shutil.rmtree(dest)
        shutil.copytree(run_root, dest)
        print('saved to Drive:', dest)
    if download_zip:
        try:
            from google.colab import files
            zip_base = Path('/content') / run_root.name
            if Path(str(zip_base) + '.zip').exists():
                Path(str(zip_base) + '.zip').unlink()
            archive = shutil.make_archive(str(zip_base), 'zip', run_root)
            files.download(archive)
        except ImportError:
            print('download_zip: не Colab')
    globals()['COMPARE_HISTORIES'] = histories
    globals()['COMPARE_RUN_DIRS'] = run_dirs
    return df


In [ ]:
results_df = train_all_models(
    hidden_size=HIDDEN_SIZE,
    dataset_dir=DATASET_DIR,
    run_root=RUNS_DIR / RUN_TAG,
    train_cfg=TRAIN_CFG,
    save_to_drive=(RUNS_DRIVE / RUN_TAG) if RUNS_DRIVE else None,
    download_zip=False,
)
display(results_df[['model_label', 'model_family', 'n_params', 'test_f1_weighted', 'test_lrap', 'train_wall_time_sec']])


## Графики обучения (overlay M0/M1/M2)

In [ ]:
def plot_training_comparison(histories: dict, out_path: Path | None = None):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for label, hist in histories.items():
        epochs = range(1, len(hist.get('train_loss', [])) + 1)
        axes[0].plot(epochs, hist['train_loss'], label=f'{label} train')
        axes[0].plot(epochs, hist['val_loss'], ls='--', label=f'{label} val')
        if 'val_lrap' in hist:
            axes[1].plot(epochs, hist['val_lrap'], label=label)
    axes[0].set_ylabel('loss')
    axes[0].legend(fontsize=7)
    axes[1].set_ylabel('val LRAP')
    axes[1].legend(fontsize=7)
    fig.suptitle('KAN vs CNN training curves')
    fig.tight_layout()
    if out_path:
        fig.savefig(out_path, dpi=140)
    plt.show()

plot_training_comparison(
    COMPARE_HISTORIES,
    out_path=RUNS_DIR / RUN_TAG / 'training_curves.png',
)


## Внешний тест: SDBS (вне dataset_v003)

In [ ]:
def load_sdbs_eval(npz_path: Path):
    z = np.load(npz_path, allow_pickle=True)
    v003_ids = set(spec_ids)
    ext_ids = [str(s) for s in z['spectrum_id']]
    overlap = [s for s in ext_ids if s in v003_ids]
    if overlap:
        print('WARNING: overlap with v003:', len(overlap))
    return z['X_input'], ext_ids, z

if SDSBS_EVAL.is_file():
    X_ext, ext_ids, sdbs_z = load_sdbs_eval(SDSBS_EVAL)
    print('external spectra:', X_ext.shape)
else:
    print('Нет', SDSBS_EVAL, '— python tools/build_sdbs_holdout.py')
    X_ext, ext_ids = None, []


## Инференс и визуализация внимания

In [ ]:
import torch.nn.functional as F
from ir_pipeline.gradcam import compute_cam, interpolate_cam_to_wavenumbers
from ir_pipeline.models.model_factory import build_spectrum_model

def load_bundle_model(bundle_path: Path):
    ck = torch.load(bundle_path, map_location='cpu', weights_only=False)
    meta = ck['meta']
    model = build_spectrum_model(
        meta.get('model_family', 'irresnet4'),
        hidden_size=int(meta['hidden_size']),
        class_nums=len(meta['class_names']),
        context_dim=int(meta.get('context_dim', 0)),
        train_cfg={
            'kan_grid_size': meta.get('kan_grid_size'),
            'kan_full_hidden_size': meta.get('kan_full_hidden_size', meta['hidden_size']),
        },
    )
    model.load_state_dict(ck['model_state'])
    model.eval()
    return model, meta

def run_external_inference(bundles, X_ext, class_names, threshold=0.5):
    rows = []
    for label, bpath in bundles.items():
        model, meta = load_bundle_model(bpath)
        with torch.no_grad():
            logits = model(torch.from_numpy(X_ext).float())
            prob = torch.sigmoid(logits).numpy()
        for i in range(len(X_ext)):
            top = np.argsort(-prob[i])[:5]
            rows.append({
                'model': label,
                'spectrum_idx': i,
                'top_bands': [class_names[j] for j in top],
                'top_probs': [float(prob[i, j]) for j in top],
            })
    return pd.DataFrame(rows)

def plot_model_attention(model, x_3ch, class_idx, wavenumbers, band_range=None):
    x = torch.from_numpy(x_3ch).float().unsqueeze(0)
    cam = compute_cam(model, x, class_idx)
    cam_i = interpolate_cam_to_wavenumbers(cam, wavenumbers)
    cam_i = cam_i / (cam_i.max() + 1e-9)
    ab = x_3ch[1]
    fig, ax = plt.subplots(figsize=(11, 3))
    ax.plot(wavenumbers, ab, color='gray', lw=1)
    ax.fill_between(wavenumbers, 0, cam_i * ab.max(), alpha=0.35, color='crimson')
    if band_range:
        ax.axvspan(band_range[0], band_range[1], color='green', alpha=0.1)
    ax.invert_xaxis()
    ax.set_xlabel(r'cm$^{-1}$')
    ax.set_title(f'Grad-CAM class_idx={class_idx}')
    fig.tight_layout()
    plt.show()
    return cam_i


In [ ]:
if X_ext is not None and 'COMPARE_RUN_DIRS' in globals():
    bundles = {k: v / 'irresnet_bundle.pt' for k, v in COMPARE_RUN_DIRS.items()}
    bundles = {k: v for k, v in bundles.items() if v.is_file()}
    if bundles:
        inf_df = run_external_inference(bundles, X_ext[:5], class_names)
        display(inf_df.head(15))
        spec_i = 0
        for label, bpath in bundles.items():
            model, meta = load_bundle_model(bpath)
            with torch.no_grad():
                logits = model(torch.from_numpy(X_ext[spec_i:spec_i+1]).float())
                cidx = int(torch.sigmoid(logits)[0].argmax())
            print('---', label, 'top class:', class_names[cidx])
            plot_model_attention(model, X_ext[spec_i], cidx, wn)
else:
    print('Пропуск инференса: нет SDBS или не обучены модели')
